# AEM4L5 E01 — Full fine-tuning vs LoRA

**Objetivo:** entender por qué adaptar modelos en producción no siempre significa reentrenar todo.

**Uso en clase:** primero explicá el problema con el gráfico, después ejecutá las celdas para cuantificar storage, tiempo y decisión.


## Idea central

Full fine-tuning actualiza todos o casi todos los pesos del modelo. LoRA/PEFT congela el modelo base e inserta adapters pequeños entrenables.

```mermaid
flowchart LR
    A["Modelo base preentrenado"] --> B["Full fine-tuning cliente A"]
    A --> C["Full fine-tuning cliente B"]
    A --> D["Full fine-tuning cliente C"]

    E["Modelo base congelado"] --> F["Adapter LoRA salud"]
    E --> G["Adapter LoRA legal"]
    E --> H["Adapter LoRA soporte"]
    F --> I["Respuesta adaptada"]
    G --> I
    H --> I
```

| Aspecto | Full fine-tuning | LoRA / PEFT |
|---|---|---|
| Parámetros entrenados | Todos o casi todos | Fracción pequeña |
| Variante por cliente | Copia completa | Adapter liviano |
| Iteración | Lenta | Rápida |
| Riesgo | Costo alto y olvido | Adapter mal evaluado |


## Ejemplo base

Una empresa tiene un modelo de resumen que ya funciona razonablemente bien. Quiere adaptarlo para 12 clientes con tono y vocabulario propio, usando una sola GPU compartida.


In [ ]:
model_gb = 14.0
adapter_gb = 0.28
clientes = 12

full_storage = model_gb * clientes
lora_storage = model_gb + adapter_gb * clientes
ahorro = 1 - (lora_storage / full_storage)

print(f"Full fine-tuning: {full_storage:.1f} GB para {clientes} clientes")
print(f"LoRA / PEFT:      {lora_storage:.1f} GB para {clientes} clientes")
print(f"Ahorro estimado:  {ahorro:.1%}")


## Criterio de decisión

LoRA no reemplaza siempre a full fine-tuning. Sirve muy bien cuando la adaptación es moderada, hay múltiples clientes o dominios, y el equipo necesita iterar rápido.


In [ ]:
def elegir_adaptacion(cambio_profundo: bool, clientes: int, gpu_limitada: bool, iteracion_semanal: bool) -> str:
    if cambio_profundo and clientes <= 1 and not gpu_limitada:
        return "Full fine-tuning"
    if clientes > 1 or gpu_limitada or iteracion_semanal:
        return "LoRA / PEFT"
    return "Evaluar con experimento pequeño"

casos = [
    {"nombre": "12 clientes, tono propio", "cambio_profundo": False, "clientes": 12, "gpu_limitada": True, "iteracion_semanal": True},
    {"nombre": "un dominio nuevo muy diferente", "cambio_profundo": True, "clientes": 1, "gpu_limitada": False, "iteracion_semanal": False},
    {"nombre": "piloto con presupuesto bajo", "cambio_profundo": False, "clientes": 3, "gpu_limitada": True, "iteracion_semanal": True},
]

for caso in casos:
    decision = elegir_adaptacion(**{k: v for k, v in caso.items() if k != "nombre"})
    print(f"{caso['nombre']}: {decision}")


## Cierre docente

La pregunta de arquitectura no es “qué técnica está de moda”, sino qué estrategia permite adaptar con calidad, costo controlado y velocidad de iteración.
